1. Instalación de librerías

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes sentencepiece
# transformes: manejo y carga de modelos LLM's pre-entrenados
# accelerate: modelo más rapido y eficiente
# bitsandbytes: uso de modelos grandes con menos memoria
# sentencepiece: en caso el modelo tiene tokenización
!pip install faiss-cpu sentence-transformers datasets PyPDF2
# faiss-cpu: busqueda vectorial para RAG
# sentence-transformers: texto a vectores que FAISS pueda buscar
# datasets: carga de información en formato manejable
# PyPDF2: permite extraer texto de PDF's

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 17.6 MB/s eta 0:00:00


2. Carga de dataset (PDF)

- Se usará un PDF que contiene la normativa de Gestión Operativa y Presupuestal para las empresas bajo el ámbito de FONAFE.

In [ ]:
from PyPDF2 import PdfReader

pdf_path = "/content/Bases_Presupuesto_FONAFE.pdf" # ruta de archivo de interés
reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

3. Chunking + overlap

- Chunking divide de forma manejable un texto largo (en chunks) a fin de que el LLM pueda buscar y entender información especifica.
- Overlap asegura que no se pierda contexto entre chunks, no perdiendo precisión en las respuestas generadas.
- Se optó por usar un chunksize de 400 y un overlap de 100 a fin de tener mejor precisión y contexto en la busqueda de información relevante para las preguntas (granularidad). No obstante, al tener más chunks, se requiere de mas memoria y tiempo de calculo para los embeddings.

In [ ]:
chunk_size = 400
overlap = 100

chunks = []

for page in text.split("\n"):
    start = 0
    while start < len(page):
        end = start + chunk_size
        chunk = page[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

print(f"Total chunks generados: {len(chunks)}") # Se cumple el requisito minimo de 100 chunks

Total chunks generados: 1287


4. Embeddings

- Cada chunk se convierte en un vector de números. La idea es que chunks con significado similar esten cerca unos de otros en el espacio vectorial.
- Sin embedding, el LLM no sabría que información es relevante para la pregunta formulada.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

5. FAISS HNSW

- HNSW es un tipo especifico de indice de busqueda dentro de FAISS. Como ya se mencionó, los embeddings de los chunks son vectores en un espacio de alta dimensión. Este indice organiza los vectores en un grafo jerarquico de nodos conectados por similitud.
- Su propósito es preparar un sistema que pueda encontrar rapidamente los chunks más semanticamente relevantes a una consulta o query (usando los embeddings del espacio vectorial), sin necesidad de revisar todos uno por uno. Es decir, se crea un sistema de busqueda eficiente.
- Las ventajas del indice HNSW son: es muy rápido incluso con miles o millones de embeddings y mantiene una alta precisión en encontrar vectores cercanos a una consulta.
- Analogía conceptual: en una biblioteca grande, si los chunks son los libros y los embeddings la ubicación de estos, el indice HNSW seria el bibliotecario que sabe donde buscar el libro más acertado para responder una consulta; en lugar de revisar cada libro hasta dar con la respuesta.

In [ ]:
import faiss
import numpy as np

d = embeddings.shape[1]
index = faiss.IndexHNSWFlat(d, 32)
index.add(np.array(embeddings))

6. Retrieval

- Ya creado el sistema de busqueda en el paso anterior, es aqui donde se realiza la misma.
- La dinamica es la siguiente:
  1. Se hace una consulta o query
  2. La consulta se convierte en embedding
  3. El indice HNSW busca los vectores más cercanos
  4. Devuelve los 5 chunks más relevantes a la consulta



In [ ]:
def retrieve(query, k=5):
    q_vec = model.encode([query])
    D, I = index.search(np.array(q_vec), k)
    return [chunks[i] for i in I[0]]

query = "¿Que se considera para realizar una modificación presupuestal?"
top_chunks = retrieve(query)

for i, chunk in enumerate(top_chunks):
    print(f"\nChunk {i+1}:\n")
    print(chunk)

print(len(chunks))


Chunk 1:

observar las modificaciones presupuestales que no cumplan con lo establecido en la 

Chunk 2:

observar las aprobaciones presupuestales que no cumplan con lo establecido en la 

Chunk 3:

presupuestales con cargo a las modificaciones presupuestales en casos que sean 

Chunk 4:

 Informe que sustente la propuesta de modificación presupuestal por cada partida y 

Chunk 5:

presupuestal, son los que se detallan a continuación:  
1287


7. Reranker cross-encoder

- Luego de que el sistema devuelva los top-5 chunks, no todos son necesariamente igual de relevantes.
- El reranker cross-encoder evalua la interacción directa entre cada chunk y la consulta: ¿Que tan relevante y contextual es este chunk con respecto al query?
- Los chunks estarán ordenados según relevancia real, mitigando ruido y aumentando la precisión de la respuesta final.

In [ ]:
from sentence_transformers import CrossEncoder

# Modelo pre-entrenado de Hugging Face
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# Prepara los pares para el reranker
pairs = [(query, chunk) for chunk in top_chunks]
# Se obtienen los scores
scores = reranker.predict(pairs)
# Combinar chunks con sus scores
reranked_chunks = list(zip(top_chunks, scores))
# Ordenar de mayor a menor score
reranked_chunks = sorted(reranked_chunks, key=lambda x: x[1], reverse=True)

for i, (chunk, score) in enumerate(reranked_chunks):
    print(f"Chunk {i+1} (score {score:.4f}):\n{chunk}\n")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Chunk 1 (score 3.2984):
observar las modificaciones presupuestales que no cumplan con lo establecido en la 

Chunk 2 (score 2.3893):
 Informe que sustente la propuesta de modificación presupuestal por cada partida y 

Chunk 3 (score 0.8393):
presupuestales con cargo a las modificaciones presupuestales en casos que sean 

Chunk 4 (score -0.3923):
presupuestal, son los que se detallan a continuación:  

Chunk 5 (score -0.5514):
observar las aprobaciones presupuestales que no cumplan con lo establecido en la 



8. LLM Qwen

- Se usa un modelo de lenguaje (LLM) para generar una respuesta a partir de una consulta. El LLM toma los chunks recuperados por el indice HNSW y, también, rerankeados según su relevancia. De ese modo, se generan respuestas basadas en información recuperada.
- Usar un modelo Qwen2-1.5B ofrece un buen equilibrio entre calidad de generación y costo computacional.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
# AutoTokenizer convierte texto en tokens númericos
# AutoModelForCausalLM carga un modelo generativo
import torch
# torch ejecuta los calculos del modelo

model_name = "Qwen/Qwen2-1.5B-Instruct" # modelo a descargar de HuggingFace

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto" # usar GPU, sino CPU
)

# Selección de 3 chunks más relevantes para responder el query
top_k = 3
context = "\n\n".join([chunk for chunk, score in reranked_chunks[:top_k]])

print("Context usado:\n")
print(context)

prompt = f"""
Explica la respuesta en una lista de puntos claros.
Cada punto debe ser una oración corta.

Contexto:
{context}

Consulta:
{query}

Respuesta:
"""

# Transformar el prompt en tokens
inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

# Generar respuesta
outputs = llm.generate(
    **inputs,
    max_new_tokens=180,
    temperature=0.1 # creatividad de respuesta, en RAG se usa 0.1-0.3
)

# Transformar tokens a texto
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

clean_response = response.split("Respuesta:")[-1].strip()

print("\nRespuesta generada:\n")
print(clean_response)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Context usado:

observar las modificaciones presupuestales que no cumplan con lo establecido en la 

 Informe que sustente la propuesta de modificación presupuestal por cada partida y 

presupuestales con cargo a las modificaciones presupuestales en casos que sean 

Respuesta generada:

Para realizar una modificación presupuestal, se deben cumplir los siguientes criterios:

1. Se necesita un informe que respalden la propuesta de modificación presupuestal por cada partida.
2. Los presupuestales con cargo a las modificaciones presupuestales también deben ser considerados.
3. Se requiere un análisis detallado de las modificaciones presupuestales para determinar si son justificadas.
4. La modificación presupuestal debe ser justificada y debidamente documentada.
5. El proceso de modificación presupuestal debe seguir los procedimientos establecidos por el organismo o entidad responsable.
6. Las modificaciones presupuestales deben ser consistentes con los objetivos y metas del organismo o en

9. Citation per sentence

- Se ejecuta lo siguiente:
  1. Se divide la respuesta generada por el LLM en oraciones.
  2. Se compara cada oración con los chunks recuperados en el punto 6 (Retrieval).
  3. Encuentra el chunk más parecido y se añade una cita "DocX".

- De ese modo se puede rastrear la fuente de informacón.

In [ ]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Limpiar respuesta
clean_response = response.split("Respuesta:")[-1]

# Dividir por saltos de linea
lines = clean_response.split("\n")

# Limpiar lineas
sentences = []

for line in lines:

    line = line.strip()

    # Eliminar lineas vacias
    if line == "":
        continue

    # Eliminar números de lista
    if re.match(r'^\d+\.?$', line):
        continue

    # Eliminar "1. texto"
    line = re.sub(r'^\d+\.\s*', '', line)

    if len(line) > 10:
        sentences.append(line)

# Embeddings de oraciones
sentence_embeddings = model.encode(sentences)

# Embeddings de chunks
chunk_texts = [chunk for chunk, score in reranked_chunks]
chunk_embeddings = model.encode(chunk_texts)

# Asignar citas
final_answer = ""

for i, sent_emb in enumerate(sentence_embeddings):

    sims = cosine_similarity([sent_emb], chunk_embeddings)[0]
    best_chunk = np.argmax(sims)

    final_answer += f"{sentences[i]} [Doc{best_chunk+1}]\n\n"

print(final_answer)

Para realizar una modificación presupuestal, se deben cumplir los siguientes criterios: [Doc1]

Se necesita un informe que respalden la propuesta de modificación presupuestal por cada partida. [Doc2]

Los presupuestales con cargo a las modificaciones presupuestales también deben ser considerados. [Doc3]

Se requiere un análisis detallado de las modificaciones presupuestales para determinar si son justificadas. [Doc1]

La modificación presupuestal debe ser justificada y debidamente documentada. [Doc2]

El proceso de modificación presupuestal debe seguir los procedimientos establecidos por el organismo o entidad responsable. [Doc2]

Las modificaciones presupuestales deben ser consistentes con los objetivos y metas del organismo o entidad. [Doc2]

La modificación presupuestal debe ser realizada de manera transparente y justa. [Doc2]




10. Hallucination guard

- Mide si la respuesta generada por el LLM esta realmente sustentada por los chunks recuperados.
- Calcula el puntaje de similitud de cada oración con los chunks.
- Puntaje bajo = posible alucinación.

In [ ]:
grounding_scores = []

for i, sent_emb in enumerate(sentence_embeddings):

    sims = cosine_similarity([sent_emb], chunk_embeddings)[0]

    best_score = np.max(sims)

    grounding_scores.append(best_score)

    if best_score < 0.4:
        print(f"⚠️ Posible alucinación: {sentences[i]}")
        print(f"Score: {best_score:.2f}\n")

grounding_score = np.mean(grounding_scores)

print("Grounding score del sistema:", round(grounding_score, 3))

Grounding score del sistema: 0.776


- La mayoría de las oraciones de la respuesta están respaldadas por los chunks recuperados. Es decir, la respuesta esta bien fundamentada.

11. Grounding evaluation

In [ ]:
grounded_sentences = 0
threshold = 0.5   # umbral para considerar grounded

for score in grounding_scores:

    if score >= threshold:
        grounded_sentences += 1

total_sentences = len(grounding_scores)

grounded_ratio = grounded_sentences / total_sentences

print("Total de oraciones:", total_sentences)
print("Oraciones grounded:", grounded_sentences)
print("Grounded ratio:", round(grounded_ratio,3))
print("Grounding score promedio:", round(np.mean(grounding_scores),3))

Total de oraciones: 8
Oraciones grounded: 8
Grounded ratio: 1.0
Grounding score promedio: 0.776


12. BLEU/ROUGE

In [ ]:
!pip install rouge-score

reference_answer = """
Para realizar una modificación presupuestal se debe presentar un informe que sustente la propuesta de modificación por cada partida.
Las modificaciones deben estar justificadas y cumplir con los requisitos establecidos.
"""

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1','rouge2','rougeL'],
    use_stemmer=True
)

scores = scorer.score(reference_answer, clean_response)

print("ROUGE-1:", round(scores['rouge1'].fmeasure,3))
print("ROUGE-2:", round(scores['rouge2'].fmeasure,3))
print("ROUGE-L:", round(scores['rougeL'].fmeasure,3))

ROUGE-1: 0.378
ROUGE-2: 0.233
ROUGE-L: 0.338
